In [1]:
# Initialisation
#  echo: false
# Recharge automatiquement les modules modifiés (évite de redémarrer le kernel)
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal as sci
from scipy.signal import hilbert
from scipy.stats import kurtosis
from scipy.fftpack import fft, fftfreq
from IPython.display import Markdown

# Cherche le dossier racine du projet (BDS_Fault_Detection) pour faire les imports depuis Sources/
dossier = Path().resolve()
for parent in (dossier, *dossier.parents):
    if "BDS_Fault_Detection" in parent.name:
        RACINE = parent
sys.path.insert(0, str(RACINE))

# Importation des fonctions du projet (dossier Sources/). Attention de l'ordre d'importation  !
from Sources import data, config, signaux, indicateurs, features, models


I0000 00:00:1794837226.772831   94134 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# Sélection du scope de données à traiter (sources, défauts, diamètres, positions, charges, canaux)
selection = data.selectionner(
    sources=("12kDE", "Normal"),
    defauts=("IR", "B", "OR"),
    diametres=(7, 14, 21), positions_or=("6",),
    charges=(0, 1, 2, 3),
    canaux={"12kDE": ("DE",), 
            "Normal": ("DE",)},
)

In [3]:
# Importation des données sélectionnées et ajout des indicateurs temporels et spectraux (ajout de ceux de Mourad)
table = features.construire_table_SpecMourad(selection, data.charger)

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

VARIABLES = ["RMS", "Pic", "Crete", "Kurtosis",
             "RMS_bande", "Env_BPFI", "Env_BPFO", "Env_BSF"]
CIBLE = "defaut"
ORDRE_LOC = ["Sain", "IR", "B", "OR"]

TEST_VAR = "charge_hp"
print("########## ======================== #########" \
"\nData : Roulement {}" \
"\n     : Tous diamètres {}" \
"\n     : Tous défauts : {} (OR006)" \
"\n\nTrain/test : apprentissage sur 3 charges, test sur la 4e" \
"\nVariation par test : {}" \
"\nVariable cible : {}" \
"\nVariables explicatives : {}" \
"\nScore : Accuracy" \
"\nModèle : Random Forest (n_estimators=300, class_weight='balanced')" \
.format(table["canal"].unique().tolist(), table["diametre_mils"].unique().tolist(), table["defaut"].unique().tolist(), TEST_VAR, CIBLE, VARIABLES))


scores, predictions = [], []
for charge_test in range(4):
    app  = table[table[TEST_VAR] != charge_test]     # apprentissage : 3 charges
    test = table[table[TEST_VAR] == charge_test]     # test : la 4e charge
    modele = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                    random_state=0)
    modele.fit(app[VARIABLES], app[CIBLE])
    pred = modele.predict(test[VARIABLES])
    scores.append({"Test N°": charge_test+1,
                    "Charge train": app[TEST_VAR].unique()[0:3].tolist(),
                   "Charge testée (HP)": charge_test,
                   "Fenêtres testées": len(test),
                   "Exactitude (%)": round(100 * accuracy_score(test[CIBLE], pred), 1)})
    predictions.append(test.assign(prediction=pred))

predictions = pd.concat(predictions)
display(pd.DataFrame(scores))
mc = pd.DataFrame(confusion_matrix(predictions[CIBLE], predictions["prediction"],
                                   labels=ORDRE_LOC),
                  index=ORDRE_LOC, columns=ORDRE_LOC)
mc.index.name = "Réel \\ Prédit"
print("Matrice de confusion")
display(mc)

########## ======================== #########
Data : Roulement ['DE']
     : Tous diamètres [0, 7, 14, 21]
     : Tous défauts : ['Sain', 'IR', 'B', 'OR'] (OR006)

Train/test : apprentissage sur 3 charges, test sur la 4e
Variation par test : charge_hp
Variable cible : defaut
Variables explicatives : ['RMS', 'Pic', 'Crete', 'Kurtosis', 'RMS_bande', 'Env_BPFI', 'Env_BPFO', 'Env_BSF']
Score : Accuracy
Modèle : Random Forest (n_estimators=300, class_weight='balanced')


,Test N°,Charge train,Charge testée (HP),Fenêtres testées,Exactitude (%)
0,1,"[1, 2, 3]",0,553,98.4
1,2,"[0, 2, 3]",1,652,100.0
2,3,"[0, 1, 3]",2,652,99.4
3,4,"[0, 1, 2]",3,654,97.9


Matrice de confusion


,Sain,IR,B,OR
Réel \ Prédit,,,,
Sain,706,0,0,0
IR,0,600,1,0
B,0,1,589,11
OR,0,0,14,589
